In [1]:
!pip install --upgrade pip --quiet
!pip install diffusers transformers accelerate controlnet-aux datasets --quiet
!pip install torch-fidelity lpips --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.8 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.

In [15]:
!pip install scikit-image opencv-python-headless --quiet

In [16]:
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler
from diffusers.utils import load_image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import os
from PIL import Image
import numpy as np
import lpips
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast 
from torch_fidelity import calculate_metrics
from skimage.metrics import structural_similarity as ssim
import cv2
import warnings
warnings.filterwarnings("ignore")

# Arguments

In [3]:
# Data
base_dir = "/kaggle/input/face-sketches-collection/small_hed-augmented_ffhq_dataset"
test_dir = base_dir

# Training and Testing
num_epochs = 20
batch_size = 4 
accumulation_steps = 8
effective_batch_size = batch_size * accumulation_steps
device = "cuda"
prompt = "a realistic photo of a human face"
controlnet_name = "lllyasviel/sd-controlnet-hed"
stable_diff_name = "botp/stable-diffusion-v1-5"
padding = "max_length"
return_tensors="pt"
scaler = GradScaler()
patience = 5  
best_eval_loss = float('inf')
best_model_path = "/kaggle/working/controlnet_best_model"
latest_model_path = "/kaggle/working/controlnet_latest_model"
generated_dir = "/kaggle/working/generated_for_metrics"
max_eval_samples = 500 

# Dataset

In [4]:
class SketchToPhotoDataset(Dataset):
    def __init__(self, hed_dir, photo_dir, max_samples=None):
        self.hed_dir = hed_dir
        self.photo_dir = photo_dir
        self.hed_files = sorted(os.listdir(hed_dir))
        self.photo_files = sorted(os.listdir(photo_dir))
        
        if max_samples and len(self.hed_files) > max_samples:
            import random
            indices = random.sample(range(len(self.hed_files)), max_samples)
            self.hed_files = [self.hed_files[i] for i in indices]
            self.photo_files = [self.photo_files[i] for i in indices]
        
        self.condition_transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor()  
        ])
        self.target_transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]) 
        ])
    
    def __len__(self):
        return len(self.hed_files)

    def __getitem__(self, idx):
        hed_path = os.path.join(self.hed_dir, self.hed_files[idx])
        photo_path = os.path.join(self.photo_dir, self.photo_files[idx])
        
        hed_image = Image.open(hed_path).convert("RGB")
        photo = Image.open(photo_path).convert("RGB")
        
        hed_image = self.condition_transform(hed_image)  
        photo = self.target_transform(photo) 
        
        return {"hed": hed_image, "photo": photo}

In [5]:
train_dataset = SketchToPhotoDataset(
    hed_dir=os.path.join(base_dir, "train", "sketches"),
    photo_dir=os.path.join(base_dir, "train", "photos"),
    max_samples=None 
)
val_dataset = SketchToPhotoDataset(
    hed_dir=os.path.join(base_dir, "val", "sketches"),
    photo_dir=os.path.join(base_dir, "val", "photos"),
    max_samples=None 
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Models

In [6]:
controlnet = ControlNetModel.from_pretrained(
    controlnet_name
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    stable_diff_name, 
    controlnet=controlnet, 
    torch_dtype=torch.float16
)

# pipe.enable_xformers_memory_efficient_attention()

for param in pipe.unet.parameters():
    param.requires_grad = False
for param in pipe.text_encoder.parameters():
    param.requires_grad = False
for param in pipe.vae.parameters():
    param.requires_grad = False

controlnet.to(torch.float32) 
for param in controlnet.parameters():
    param.requires_grad = True

pipe.to(device) 
controlnet.to(device) 

optimizer = torch.optim.AdamW(controlnet.parameters(), lr=5e-6, weight_decay=1e-4) 
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)

config.json:   0%|          | 0.00/920 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/1.45G [00:00<?, ?B/s]

model_index.json:   0%|          | 0.00/543 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

# Training

In [7]:
patience_counter = 0

for epoch in range(num_epochs):
    controlnet.train()
    epoch_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch} Training")
    
    for step, batch in enumerate(progress_bar):
        hed_images = batch["hed"].to(device)
        photos = batch["photo"].to(device)
        
        with autocast():
            with torch.no_grad():
                latents = pipe.vae.encode(photos).latent_dist.sample() * pipe.vae.config.scaling_factor
            
            # timesteps and noise
            bsz = latents.shape[0]
            timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=device).long()
            noise = torch.randn_like(latents)
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            
            # Encode prompt
            use_null_prompt = torch.rand(1).item() < 0.1
            
            if use_null_prompt:
                final_prompt = "" 
            else:
                final_prompt = prompt
                
            text_inputs = pipe.tokenizer(
                final_prompt, 
                padding=padding, 
                max_length=pipe.tokenizer.model_max_length, 
                truncation=True, 
                return_tensors=return_tensors
            )
        
            text_input_ids = text_inputs.input_ids.to(device)
            
            with torch.no_grad():
                encoder_hidden_states = pipe.text_encoder(text_input_ids)[0]
                if encoder_hidden_states.shape[0] != bsz:
                    encoder_hidden_states = encoder_hidden_states.repeat(bsz, 1, 1)

            # Forward ControlNet
            controlnet_output = controlnet(
                sample=noisy_latents,
                timestep=timesteps,
                encoder_hidden_states=encoder_hidden_states,
                controlnet_cond=hed_images,
                return_dict=True 
            )
            
            down_block_res_samples = controlnet_output.down_block_res_samples
            mid_block_res_sample = controlnet_output.mid_block_res_sample
            
            # Forward UNet
            noise_pred = pipe.unet(
                noisy_latents, 
                timestep=timesteps, 
                encoder_hidden_states=encoder_hidden_states, 
                down_block_additional_residuals=down_block_res_samples, 
                mid_block_additional_residual=mid_block_res_sample
            ).sample
            
            # loss 
            loss = torch.nn.functional.mse_loss(noise_pred, noise)
            epoch_loss += loss.item()
            
            loss = loss / accumulation_steps
        
        scaler.scale(loss).backward()
        
        if (step + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update() 
            optimizer.zero_grad()
        
        progress_bar.set_postfix(Loss=f"{loss.item() * accumulation_steps:.4f}")
    
    avg_train_loss = epoch_loss / len(train_dataloader)
    print(f"\nEpoch {epoch}, Avg Train Loss: {avg_train_loss:.4f}")

    # Validation
    controlnet.eval()
    val_loss = 0
    val_progress_bar = tqdm(val_dataloader, desc=f"Epoch {epoch} Validation")
    
    with torch.no_grad():
        for batch in val_progress_bar:
            hed_images = batch["hed"].to(device)
            photos = batch["photo"].to(device)
            
            with autocast(): 
                latents = pipe.vae.encode(photos).latent_dist.sample() * pipe.vae.config.scaling_factor
                bsz = latents.shape[0]
                timesteps = torch.randint(0, pipe.scheduler.config.num_train_timesteps, (bsz,), device=device).long()
                noise = torch.randn_like(latents)
                noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
                
                text_inputs = pipe.tokenizer(
                    prompt, 
                    padding=padding, 
                    max_length=pipe.tokenizer.model_max_length, 
                    truncation=True, 
                    return_tensors=return_tensors
                )
                
                text_input_ids = text_input_ids.to(device)
                
                encoder_hidden_states = pipe.text_encoder(text_input_ids)[0]
                
                if encoder_hidden_states.shape[0] != bsz:
                    encoder_hidden_states = encoder_hidden_states.repeat(bsz, 1, 1)
                
                controlnet_output = controlnet(
                    sample=noisy_latents,
                    timestep=timesteps,
                    encoder_hidden_states=encoder_hidden_states,
                    controlnet_cond=hed_images,
                    return_dict=True
                )
                
                noise_pred = pipe.unet(
                    noisy_latents, 
                    timestep=timesteps, 
                    encoder_hidden_states=encoder_hidden_states, 
                    down_block_additional_residuals=controlnet_output.down_block_res_samples, 
                    mid_block_additional_residual=controlnet_output.mid_block_res_sample
                ).sample
                
                val_loss += torch.nn.functional.mse_loss(noise_pred, noise).item()
            
            val_progress_bar.set_postfix(Val_Loss=f"{val_loss / len(val_dataloader):.4f}")
            
    avg_val_loss = val_loss / len(val_dataloader)
    print(f"Epoch {epoch}, Avg Val Loss: {avg_val_loss:.4f}")
    
    # Early stopping
    if avg_val_loss < best_eval_loss:
        best_eval_loss = avg_val_loss
        patience_counter = 0
        
        controlnet.save_pretrained(best_model_path)
        print(f"Saved best model at: {best_model_path}")
        
    else:
        patience_counter += 1
        print(f"Patience: {patience_counter} / {patience}")
        
        if patience_counter >= patience:
            print(f"Early stopping after {patience} epochs.")
            break 
    
    scheduler.step()

controlnet.save_pretrained(latest_model_path)
print(f"Saved best model (Eval Loss: {best_eval_loss:.4f}) at: {best_model_path}")
print(f"Saved final model at: {latest_model_path}")

Epoch 0 Training: 100%|██████████| 707/707 [33:15<00:00,  2.82s/it, Loss=0.1709]



Epoch 0, Avg Train Loss: 0.1354


Epoch 0 Validation: 100%|██████████| 40/40 [00:54<00:00,  1.37s/it, Val_Loss=0.1379]


Epoch 0, Avg Val Loss: 0.1379
Saved best model at: /kaggle/working/controlnet_best_model


Epoch 1 Training: 100%|██████████| 707/707 [32:23<00:00,  2.75s/it, Loss=0.0829]


...

Epoch 15, Avg Val Loss: 0.1480
Patience: 4 / 5


Epoch 16 Training: 100%|██████████| 707/707 [32:37<00:00,  2.77s/it, Loss=0.0451]



Epoch 16, Avg Train Loss: 0.1305


Epoch 16 Validation: 100%|██████████| 40/40 [00:49<00:00,  1.23s/it, Val_Loss=0.1334]


Epoch 16, Avg Val Loss: 0.1334
Patience: 5 / 5
Early stopping after 5 epochs.
Saved best model (Eval Loss: 0.1063) at: /kaggle/working/controlnet_best_model
Saved final model at: /kaggle/working/controlnet_latest_model


In [8]:
!zip -r -q /kaggle/working/controlnet_best_model.zip /kaggle/working/controlnet_best_model

# Testing

In [5]:
!pip install -q gdown

In [6]:
import gdown

url = 'https://drive.google.com/drive/folders/1Iu47gM9GPrirHV50QSvTEsbd7TM0mE-o?'

gdown.download_folder(url, output=best_model_path, quiet=True)

['/kaggle/working/controlnet_best_model/config.json',
 '/kaggle/working/controlnet_best_model/diffusion_pytorch_model.safetensors']

In [7]:
real_dir = os.path.join(test_dir, "test", "photos")
sketch_dir = os.path.join(test_dir, "test", "sketches")

os.makedirs(generated_dir, exist_ok=True)

generator = torch.Generator(device=device).manual_seed(1234)

In [8]:
prompt = """(hyper-realistic photo:1.2), (ultra-detailed skin texture:1.1), 
            detailed pores, realistic eyes, sharp focus, 
            8k UHD, professional studio lighting, DSLR"""

negative_prompt = """(drawing:1.4), (sketch:1.4), (painting:1.3), cartoon, 3D, 
                    render, CGI, anime, illustration, (deformed:1.2), (disfigured:1.2), 
                    ugly, bad anatomy, (blurry:1.1), low quality, low-res"""

In [9]:
controlnet = ControlNetModel.from_pretrained(
    best_model_path, 
    torch_dtype=torch.float16
)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    stable_diff_name, 
    controlnet=controlnet, 
    torch_dtype=torch.float16
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.to(device)

loss_fn_vgg = lpips.LPIPS(net='vgg').to(device)

model_index.json:   0%|          | 0.00/543 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

...

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:02<00:00, 204MB/s] 


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


### LPIPS


In [10]:
lpips_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

total_lpips_distance = 0
image_count = 0

sketch_files = sorted(os.listdir(sketch_dir))

In [11]:
for i, filename in enumerate(tqdm(sketch_files)):
    if max_eval_samples and i >= max_eval_samples:
        break

    sketch_path = os.path.join(sketch_dir, filename)
    real_path = os.path.join(real_dir, filename)
    generated_path = os.path.join(generated_dir, filename)

    condition_image = load_image(sketch_path).resize((512, 512))

    generated_image_pil = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=condition_image,
        num_inference_steps=30,
        generator=generator,
        guidance_scale=7.5,
        controlnet_conditioning_scale=0.9
    ).images[0]
    
    generated_image_pil.save(generated_path)
    real_image_pil = load_image(real_path)
    real_tensor = lpips_transform(real_image_pil).to(device)
    gen_tensor = lpips_transform(generated_image_pil).to(device)

    with torch.no_grad():
        dist = loss_fn_vgg(real_tensor.unsqueeze(0), gen_tensor.unsqueeze(0))
    
    total_lpips_distance += dist.item()
    image_count += 1

  0%|          | 0/158 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  1%|          | 1/158 [00:12<32:53, 12.57s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

  1%|▏         | 2/158 [00:24<31:27, 12.10s/it]

...

 99%|█████████▊| 156/158 [30:41<00:23, 11.80s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

 99%|█████████▉| 157/158 [30:53<00:11, 11.81s/it]

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 158/158 [31:05<00:00, 11.80s/it]


In [14]:
avg_lpips = total_lpips_distance / image_count

print(f"Average LPIPS: {avg_lpips:.4f}")

Average LPIPS: 0.5999


### FID and KID

In [15]:
metrics = calculate_metrics(
    input1=real_dir,
    input2=generated_dir,
    cuda=True,
    fid=True,
    kid=True,
    input1_max_samples=image_count,
    input2_max_samples=image_count,
    kid_subset_size=image_count
)

print(f"FID: {metrics['frechet_inception_distance']:.4f}")
print(f"KID Mean: {metrics['kernel_inception_distance_mean']:.4f}")
print(f"KID Std: {metrics['kernel_inception_distance_std']:.4f}")

Creating feature extractor "inception-v3-compat" with features ['2048']
Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:00<00:00, 98.3MB/s]
Extracting features from input1
Looking for samples non-recursivelty in "/kaggle/input/face-sketches-collection/small_hed-augmented_ffhq_dataset/test/photos" with extensions png,jpg,jpeg
Found 158 samples
Processing samples                                                        
Extracting features from input2
Looking for samples non-recursivelty in "/kaggle/working/generated_for_metrics" with extensions png,jpg,jpeg
Found 158 samples
Processing samples                                                         
Frechet Inception Distance: 164.87163437912574
                                                                                 

FID: 164.8716
KID Mean: 0.0794
KID Std: 0.0000


Kernel Inception Distance: 0.0793692014630051 ± 2.1564507138173977e-07


### SSIM

In [12]:
grayscale_gen_dir = "/kaggle/working/grayscale_generated_dir"
grayscale_real_dir = "/kaggle/working/grayscale_real_dir"

os.makedirs(grayscale_gen_dir, exist_ok=True)
os.makedirs(grayscale_real_dir, exist_ok=True)

In [13]:
def calculate_ssim(real_path_source, gen_path_source):
    scores = []
    filenames = sorted(os.listdir(gen_path_source))
    
    for filename in tqdm(filenames, desc="Processing SSIM"):
        path_real = os.path.join(real_path_source, filename)
        path_gen = os.path.join(gen_path_source, filename)
        
        if os.path.exists(path_real) and os.path.exists(path_gen):
            img_real = cv2.imread(path_real)
            img_gen = cv2.imread(path_gen)
            
            if img_real is None or img_gen is None:
                continue
                
            if img_real.shape != img_gen.shape:
                img_real = cv2.resize(img_real, (img_gen.shape[1], img_gen.shape[0]))

            img_real_gray = cv2.cvtColor(img_real, cv2.COLOR_BGR2GRAY)
            img_gen_gray = cv2.cvtColor(img_gen, cv2.COLOR_BGR2GRAY)
            
            save_path_real_gray = os.path.join(grayscale_real_dir, filename)
            save_path_gen_gray = os.path.join(grayscale_gen_dir, filename)
            
            cv2.imwrite(save_path_real_gray, img_real_gray)
            cv2.imwrite(save_path_gen_gray, img_gen_gray)
            
            score = ssim(img_real_gray, img_gen_gray, data_range=255)
            scores.append(score)
            
    return np.mean(scores)

In [17]:
current_ssim = calculate_ssim(real_dir, generated_dir)

print(f"Average SSIM: {current_ssim:.4f}")

Processing SSIM: 100%|██████████| 158/158 [00:08<00:00, 18.71it/s]

Average SSIM: 0.2770


In [18]:
!zip -r -q /kaggle/working/generated_for_metrics.zip /kaggle/working/generated_for_metrics
!zip -r -q /kaggle/working/grayscale_generated_dir.zip /kaggle/working/grayscale_generated_dir
!zip -r -q /kaggle/working/grayscale_real_dir.zip /kaggle/working/grayscale_real_dir